# Этап 8 V1 — FT-Transformer

## Исследовательский вопрос

Может ли FT-Transformer на **тех же 47 разрешённых признаках** превзойти принятый базовый ориентир (B^*=GBDT\_mean) по outer-OOF Gini?

Это отдельный контролируемый эксперимент после Stage 6 (отдельная TabM) и Stage 7 (TabM поверх GBDT). Меняется только семейство нейросетевой модели: вместо TabM используется официальный FT-Transformer из `rtdl_revisiting_models==0.0.2`. Данные, допустимые признаки, outer-CV, inner-разбиение, seeds, метрики, порог 0.5 и запрет final test остаются неизменными.

Результат будет считаться доказательством улучшения только если основная метрика — outer-OOF Gini — превысит (B^*).


## Почему зафиксирована именно эта конфигурация?

| Решение | Зафиксированное значение | Причина |
|---|---:|---|
| Реализация | `rtdl_revisiting_models.FTTransformer==0.0.2` | Официальная рекомендуемая реализация авторов RTDL; кастомный Transformer не используется. |
| Непрерывные признаки | 47, категориальных нет | Точно тот же зафиксированный контракт признаков Stage 1/6/7. |
| Токен / блок | `d_token=d_block=192`, 3 блока, 8 heads | Значения зафиксированы EXPERIMENT LOCK. |
| FFN | ReGLU, 256 hidden, dropout 0.10 | Нативная ReGLU-FFN пакета; (192×4/3=256), первая проекция ReGLU имеет 512 каналов. |
| Attention | dropout 0.20, residual dropout 0 | Регуляризация attention без дополнительного residual stochasticity. |
| Оптимизация | AdamW из `make_parameter_groups()`, lr 1e-4, wd 1e-5 | Официальный способ разделения параметров для weight decay. |
| Batch size | 256 | Заранее зафиксированный official/default-compatible вариант из EXPERIMENT LOCK; не выбирался по KOMUS OOF и не является tuning. |
| Предобработка | QuantileTransformer→normal + шум train 1e-3 | Official RTDL recipe; fit только на inner/refit train. |
| Выбор | inner ROC-AUC, patience 16 | Не использует outer validation для выбора эпохи. |
| Baseline | (B^*=GBDT\_mean) | Уже принятый простой арифметический mean Stage 7; GBDT не переобучается. |

Гиперпараметры не подбираются и не тюнятся. Linformer/KV compression отключены. Mixed precision, torch.compile, scheduler, warmup, gradient clipping, class weights и sampling не применяются.


## Предобработка, leakage и устройство эксперимента

### 1. Данные и контракт признаков

Ноутбук читает зафиксированные metadata Stage 1, сверяет хеши dataset/index/features и использует ровно 47 допустимых исходных признаков. Цель — `DefMark`, идентификатор — `INN`; `Q_B1_norm` и `Q_B2_norm` запрещены. Final test отделяется сразу и нигде не передаётся в обучение, selection, OOF или decision.

### 2. Outer / inner protocol

Outer CV — StratifiedKFold(3, shuffle=True, random_state=42). Для каждого outer-fold применяется только `outer_train`: внутренний StratifiedShuffleSplit использует 90% train / 10% validation с seed outer fold 43 / 44 / 45. Ранняя остановка выбирает лучшую эпоху по ROC-AUC; затем новый estimator refit-ится на полном outer_train ровно `best_epoch` эпох.

### 3. QuantileTransformer без leakage

Для каждого inner/refit training subset создаётся новый `QuantileTransformer(output_distribution="normal", n_quantiles=1000, random_state=seed, subsample=1_000_000_000)`. Большой официальный sentinel исключает subsampling. Шум 1e-3 добавляется только к копии train-матрицы, на которой происходит fit; исходные train/validation/outer-validation не шумятся. Transform обучен только на соответствующем train.

### 4. Метрики и решение

Outer-OOF собирается один раз на каждом outer validation. B*=GBDT_mean читается из Stage 7 OOF artifact и не переобучается. Locked decision: `material_gain` только при ΔGini >= +0.010 и превосходстве минимум на 2 из 3 folds; `inferior` только при ΔGini <= -0.010 и проигрыше минимум на 2 folds; иначе `no_material_benefit`.


### Что проверяем?

Окружение, identity эксперимента и детерминированный CPU-путь.

### Зачем сейчас?

До обучения нужно зафиксировать воспроизводимый контракт.

### Как это отвечает на исследовательский вопрос?

Результат будет сопоставим с B* только при неизменном protocol.

### Что остаётся неизменным?

Features, CV, seeds, architecture, preprocessing и final-test policy.


# Этап 8 V1 — FT-Transformer против GBDT_mean

## Исследовательский вопрос

**Может ли FT-Transformer на тех же 47 разрешённых признаках дать существенное улучшение outer-OOF Gini относительно принятого базового ориентира `B*=GBDT_mean`?**

Улучшение оценивается не по отдельному фолду и не по одной вспомогательной метрике, а по заранее зафиксированному правилу на полном outer-OOF.

## Почему проверяем это сейчас?

На предыдущих этапах уже исследовалось применение TabM. Полный Stage 6 показал, что самостоятельная TabM не превзошла сильную бустинговую модель при том же 47-признаковом протоколе.

К началу Stage 8 в качестве базового ориентира уже зафиксирован `B*=GBDT_mean` из Stage 7.

Следующий вопрос — способен ли другой современный подход для табличных данных, FT-Transformer, извлечь из тех же разрешённых признаков дополнительную прогностическую информацию без изменения данных и правил оценки.

Таким образом, здесь проверяется не новый набор данных и не новый target, а другое семейство модели.

## Что именно меняется?

Меняется исследуемый model pipeline:

- используется официальный `FTTransformer` из `rtdl_revisiting_models==0.0.2`;
- для FT-Transformer применяется заранее зафиксированная fold-local предобработка `QuantileTransformer → normal`;
- шум `1e-3` используется только при fit преобразования на копии соответствующего training subset;
- выбор числа эпох выполняется на внутренней validation-части по `ROC-AUC`;
- после выбора `best_epoch` модель заново обучается на полном outer-train ровно это число эпох.

Сохранённый `GBDT_mean` используется только как контрольный OOF baseline и заново не обучается.

## Что остаётся неизменным?

Основные условия сравнения зафиксированы заранее:

- dataset: `Data_final.xlsb`;
- SHA-256 dataset: `fc742be66d238c529daba52ccc755f774f836b7d052ed062cdf0b345080e7930`;
- target: `DefMark`;
- identifier: `INN`;
- рабочая выборка и её порядок;
- ровно 47 разрешённых признаков;
- `Q_B1_norm` и `Q_B2_norm` не используются как predictors;
- outer CV: `StratifiedKFold`, 3 folds, `shuffle=True`, `random_state=42`;
- seeds внешних фолдов: `43 / 44 / 45`;
- набор рассчитываемых метрик;
- порог `0.5` используется только для диагностических `Precision`, `Recall` и `F1`;
- class weights, balancing, sampling и threshold optimization не применяются;
- final test не используется ни для обучения, ни для выбора настроек, ни для research decision.

## Почему выбраны именно такие настройки?

Конфигурация FT-Transformer была зафиксирована до просмотра Stage 8 OOF-результата.

Основные параметры:

| Параметр | Значение |
|---|---:|
| Реализация | `rtdl_revisiting_models==0.0.2` |
| Архитектура | `FTTransformer` |
| Число признаков | 47 |
| `d_token / d_block` | 192 / 192 |
| Transformer blocks | 3 |
| Attention heads | 8 |
| Attention dropout | 0.20 |
| FFN hidden | 256 |
| FFN dropout | 0.10 |
| Optimizer | AdamW |
| Learning rate | `1e-4` |
| Weight decay | `1e-5` |
| Batch size | 256 |
| Early stopping patience | 16 |
| Selection metric | `ROC-AUC` |
| QuantileTransformer | 1000 quantiles, normal output |

Первоначальный compute ceiling составлял 1000 эпох. До получения Stage 8 OOF-результата он был уменьшен до 100 эпох из-за стоимости CPU-only запуска.

Это изменение отдельно сохранено как pre-run protocol amendment и не основывалось на качестве FT-Transformer на данных KOMUS.

Остальные существенные условия эксперимента при этом не менялись.

### Правило принятия решения

Заранее зафиксировано:

- `material_gain` — только если полный `ΔGini >= +0.010` и FT-Transformer выигрывает минимум на 2 из 3 folds;
- `inferior` — только если полный `ΔGini <= -0.010` и FT-Transformer проигрывает минимум на 2 из 3 folds;
- во всех остальных случаях — `no_material_benefit`.

In [1]:
from __future__ import annotations

import copy
from collections.abc import Mapping
import hashlib
import json
import os
import random
import tempfile
from pathlib import Path
from typing import TypedDict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from rtdl_revisiting_models import FTTransformer
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import QuantileTransformer
from torch.utils.data import DataLoader, TensorDataset

workspace_root = Path.cwd()
if not (workspace_root / "reports").exists():
    workspace_root = workspace_root.parent
GENERATED = workspace_root / "reports" / "generated"
SUMMARY = workspace_root / "reports" / "summary"
DATASET = workspace_root / "data" / "raw" / "Data_final.xlsb"
STAGE1_PATH = GENERATED / "stage1_baseline_results_V2.json"
STAGE7_PATH = GENERATED / "stage7_tabm_stacking_results_V1.json"
STAGE7_OOF_PATH = GENERATED / "stage7_tabm_stacking_oof_V1.npz"
RESULT_PATH = GENERATED / "stage8_ft_transformer_results_V1.json"
OOF_PATH = GENERATED / "stage8_ft_transformer_oof_V1.npz"
SUMMARY_PATH = SUMMARY / "stage8_ft_transformer_summary_V1.json"
CHECKPOINT_PATH = GENERATED / "stage8_ft_transformer_checkpoint_V1.pt"

TARGET, IDENTIFIER = "DefMark", "INN"
FORBIDDEN = ("Q_B1_norm", "Q_B2_norm")
OUTER_SEED, OUTER_FOLDS, OUTER_FOLD_SEEDS = 42, 3, (43, 44, 45)
THRESHOLD, NUM_WORKERS = 0.5, 0
DEVICE = torch.device("cpu")
torch.use_deterministic_algorithms(True)

FT_CONFIG = {
    "library": "rtdl_revisiting_models==0.0.2",
    "arch_type": "FTTransformer",
    "n_cont_features": 47, "cat_cardinalities": [], "d_out": 1,
    "d_token": 192, "d_block": 192, "n_blocks": 3,
    "attention_n_heads": 8, "attention_dropout": 0.20,
    "ffn_activation": "ReGLU (package-native)", "ffn_d_hidden": 256,
    "ffn_d_hidden_multiplier": 4 / 3, "ffn_first_projection": 512,
    "ffn_dropout": 0.10, "residual_dropout": 0.0,
    "linformer_kv_compression": None, "initialization": "official native defaults",
    "optimizer": "AdamW(model.make_parameter_groups())", "lr": 1e-4,
    "weight_decay": 1e-5, "betas": [0.9, 0.999], "eps": 1e-8,
    "batch_size": 256, "num_workers": 0, "max_epochs": 100,
    "amp": False, "torch_compile": False, "scheduler": None, "warmup": None,
    "gradient_clip_global_norm": None, "class_weights": None, "sampling": None,
}
PREPROCESSING_CONTRACT = {"transformer": "QuantileTransformer", "n_quantiles": 1000,
    "output_distribution": "normal", "subsample": 1_000_000_000, "training_noise": 1e-3,
    "fit_policy": "fold-local fit только на соответствующем train; шум только на копии train"}
SELECTION_CONTRACT = {
    "inner_train_fraction": 0.90, "inner_validation_fraction": 0.10,
    "patience": 16, "min_delta": 0.0, "selection_metric": "ROC-AUC",
    "refit_epochs": "best_epoch", "outer_fold_seeds": list(OUTER_FOLD_SEEDS),
}
class RngState(TypedDict):
    python: tuple[object, ...]
    numpy: tuple[str, np.ndarray, int, int, float]
    torch: torch.Tensor
    loader: torch.Tensor

class CheckpointState(TypedDict):
    contract: Mapping[str, object]
    fold: int
    phase: str
    epoch: int
    runtime_before_session: float
    fold_results: list[dict[str, object]]
    oof: np.ndarray
    best_auc: float
    best_epoch: int
    stale: int
    model: Mapping[str, torch.Tensor]
    optimizer: dict[str, dict[str, object]]
    rng: RngState
    runtime_breakdown: dict[str, dict[str, float]]
    phase_runtime_before: float
    fold_runtime_before: float
    phase_started: float
    fold_started: float
    run_mode: str
    last_checkpoint: str | dict[str, object]
    _runtime_base: float
    _runtime_session_started: float

LIMITATIONS = [
    "random CV не доказывает temporal stability",
    "3 folds не являются statistical significance claim",
    "FT-Transformer не доказывает business benefit",
    "threshold 0.5 диагностический",
    "final test не использован",
    "результат относится только к locked Stage 8 FT-Transformer design",
]

def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()

def atomic_json(path: Path, payload: Mapping[str, object]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=path.parent, delete=False) as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)
        temp_name = handle.name
    os.replace(temp_name, path)

def atomic_torch(path: Path, payload: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=path.parent, delete=False, suffix=".pt") as handle:
        temp_path = handle.name
    torch.save(payload, temp_path); os.replace(temp_path, path)

def atomic_npz(
    path: Path, working_indices: np.ndarray, target: np.ndarray, fold: np.ndarray,
    ft_transformer_oof_probability: np.ndarray, gbdt_mean_probability: np.ndarray,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + ".tmp.npz")
    np.savez_compressed(
        temp_path, working_indices=working_indices, target=target, fold=fold,
        ft_transformer_oof_probability=ft_transformer_oof_probability, gbdt_mean_probability=gbdt_mean_probability,
    )
    os.replace(temp_path, path)

def metrics(y_true: np.ndarray, probability: np.ndarray) -> dict[str, float]:
    prediction = (probability >= THRESHOLD).astype(np.int8)
    roc_auc = float(roc_auc_score(y_true, probability))
    has_positive_prediction = bool(np.any(prediction == 1))
    has_positive_target = bool(np.any(y_true == 1))
    precision = float(precision_score(y_true, prediction)) if has_positive_prediction else 0.0
    recall = float(recall_score(y_true, prediction)) if has_positive_target else 0.0
    f1 = float(f1_score(y_true, prediction)) if has_positive_prediction and has_positive_target else 0.0
    return {
        "roc_auc": roc_auc, "gini": 2.0 * roc_auc - 1.0,
        "pr_auc": float(average_precision_score(y_true, probability)),
        "precision": precision, "recall": recall, "f1": f1,
    }

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)


### Что проверяем?

Проверяем Stage 1/Stage 7 provenance, hashes, порядок 47 признаков и B*=GBDT_mean.

### Зачем сейчас?

Это останавливает run до обучения при любой несовместимости исходных данных.

### Как это отвечает на исследовательский вопрос?

Сравнение FT-Transformer с B* остаётся честным только на том же working sample.

### Что остаётся неизменным?

Final test не используется; Stage 1–7 не изменяются.


In [2]:
stage1 = json.loads(STAGE1_PATH.read_text(encoding="utf-8"))
stage7 = json.loads(STAGE7_PATH.read_text(encoding="utf-8"))
stage7_oof = np.load(STAGE7_OOF_PATH, allow_pickle=False)

assert FT_CONFIG["n_cont_features"] == len(stage1["допустимые_признаки"]) == 47
assert stage1["target"] == TARGET
assert stage1["identifier"] == IDENTIFIER
assert tuple(stage1["исключённые_признаки"]) == FORBIDDEN
assert stage7["baseline_selection"]["B_star"] == "GBDT_mean"
assert stage7["raw_features_in_order"] == stage1["допустимые_признаки"]
assert stage7["final_test_used"] is False
assert {"working_indices", "target", "gbdt_mean"}.issubset(stage7_oof.files)

data = pd.read_excel(DATASET, engine="pyxlsb", sheet_name="Data_final").reset_index(drop=True)
features = list(stage1["допустимые_признаки"])
assert data[features].shape[1] == 47 and not bool(np.any(data[features].isna().to_numpy()))
dataset_hash = sha256_bytes(DATASET.read_bytes())
assert dataset_hash == stage1["dataset"]["sha256"], "STOP: dataset hash не совпадает с locked Stage 1"

working_index = np.asarray(stage7_oof["working_indices"], dtype=np.int64)
y_working = np.asarray(stage7_oof["target"], dtype=np.int8)
p_baseline = np.asarray(stage7_oof["gbdt_mean"], dtype=np.float64)
assert sha256_bytes(working_index.astype(np.int64).tobytes()) == stage7["working_index_sha256"]
assert len(working_index) == len(y_working) == len(p_baseline)
assert np.array_equal(data.loc[working_index, TARGET].to_numpy(np.int8), y_working)
assert np.isclose(metrics(y_working, p_baseline)["gini"], stage7["baseline_selection"]["gbdt_mean_metrics"]["Gini"], atol=1e-12)

x_working = data.loc[working_index, features].to_numpy(dtype=np.float64, copy=True)
del data  # final test и вся остальная выборка дальше не используются
print(f"Предварительная проверка пройдена: {x_working.shape[0]} рабочих строк, 47 признаков; final test не использован.")


Предварительная проверка пройдена: 289614 рабочих строк, 47 признаков; final test не использован.


### Что проверяем?

FT-Transformer factory и fold-local preprocessing.

### Зачем сейчас?

Чтобы исключить leakage и сохранить locked recipe.

### Как это отвечает на исследовательский вопрос?

Каждый fold получает сопоставимый вход модели.

### Что остаётся неизменным?

Архитектура и preprocessing из lock.

## Точный RTDL-рецепт и FT-Transformer factory

Следующая ячейка намеренно отделяет fit QuantileTransformer от transform. Даже при возобновлении чекпоинта preprocessing детерминированно воспроизводится из train subset и fold seed. Пакетная модель получает `x_cat=None`: категориальных признаков по контракту нет.


In [3]:
def quantile_fit_transform(
    x_train: np.ndarray, x_apply: np.ndarray, seed: int
) -> tuple[np.ndarray, np.ndarray, QuantileTransformer]:
    if not (np.isfinite(x_train).all() and np.isfinite(x_apply).all()):
        raise ValueError("STOP: QuantileTransformer получил non-finite значения.")
    # Официальный RTDL recipe: noise только на train-copy для fit, без subsampling.
    fit_train = x_train.astype(np.float64, copy=True)
    std = np.std(fit_train, axis=0, keepdims=True)
    noise_std = 1e-3 / np.maximum(std, 1e-3)
    fit_train += noise_std * np.random.default_rng(seed).standard_normal(fit_train.shape)
    transformer = QuantileTransformer(
        n_quantiles=1000, output_distribution="normal",
        subsample=1_000_000_000, random_state=seed,
    )
    transformer.fit(fit_train)
    return (
        np.asarray(transformer.transform(x_train), dtype=np.float32),
        np.asarray(transformer.transform(x_apply), dtype=np.float32),
        transformer,
    )

def make_model() -> FTTransformer:
    return FTTransformer(
        n_cont_features=47, cat_cardinalities=[], d_out=1,
        n_blocks=3, d_block=192, attention_n_heads=8,
        attention_dropout=0.20, ffn_d_hidden=None,
        ffn_d_hidden_multiplier=4 / 3, ffn_dropout=0.10,
        residual_dropout=0.0, linformer_kv_compression_ratio=None,
        linformer_kv_compression_sharing=None,
    ).to(DEVICE)

def make_optimizer(model: FTTransformer) -> torch.optim.AdamW:
    return torch.optim.AdamW(
        model.make_parameter_groups(), lr=1e-4, weight_decay=1e-5,
        betas=(0.9, 0.999), eps=1e-8,
    )

def optimizer_state_by_name(model: FTTransformer, optimizer: torch.optim.AdamW) -> dict[str, dict[str, object]]:
    names = {id(parameter): name for name, parameter in model.named_parameters()}
    return {"state": {names[id(parameter)]: copy.deepcopy(value) for parameter, value in optimizer.state.items()}}

def restore_optimizer_state(
    model: FTTransformer, payload: Mapping[str, object]
) -> torch.optim.AdamW:
    optimizer = make_optimizer(model)
    parameters = dict(model.named_parameters())
    saved_states = payload.get("state")
    if not isinstance(saved_states, Mapping):
        raise RuntimeError("STOP: checkpoint не содержит корректного состояния optimizer.")
    if set(saved_states) - set(parameters):
        raise RuntimeError("STOP: checkpoint содержит неизвестные параметры optimizer.")
    for name, saved_state in saved_states.items():
        if not isinstance(name, str) or not isinstance(saved_state, Mapping):
            raise RuntimeError("STOP: checkpoint содержит некорректное состояние optimizer.")
        optimizer.state[parameters[name]] = copy.deepcopy(dict(saved_state))
    return optimizer

def predict(model: FTTransformer, x: np.ndarray) -> np.ndarray:
    model.eval()
    predictions: list[np.ndarray] = []

    with torch.inference_mode():
        for start in range(0, len(x), 256):
            batch = torch.from_numpy(x[start:start + 256]).to(DEVICE)
            batch_prediction = torch.sigmoid(
                model(batch, None).squeeze(1)
            ).cpu().numpy()
            predictions.append(batch_prediction)

    if not predictions:
        return np.empty(0, dtype=np.float32)

    return np.concatenate(predictions, axis=0)

def train_one_epoch(
    model: FTTransformer, optimizer: torch.optim.AdamW,
    loader: DataLoader[tuple[torch.Tensor, ...]],
) -> None:
    model.train()
    for batch_x, batch_y in loader:
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_x.to(DEVICE), None).squeeze(1)
        loss = F.binary_cross_entropy_with_logits(logits, batch_y.to(DEVICE))
        loss.backward()
        optimizer.step()

def loader_for(
    x: np.ndarray, y: np.ndarray, seed: int
) -> tuple[DataLoader[tuple[torch.Tensor, ...]], torch.Generator]:
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(x), torch.from_numpy(y.astype(np.float32))),
        batch_size=256, shuffle=True, generator=generator, num_workers=NUM_WORKERS,
    )
    return loader, generator


### Что проверяем?

Runtime, checkpoint identity и progress panel.

### Зачем сейчас?

Чтобы resume не менял evidence и не терял время.

### Как это отвечает на исследовательский вопрос?

Получается проверяемый OOF experiment trail.

### Что остаётся неизменным?

CV, seeds, decision rule и final-test policy.

## Чекпоинт/возобновление, прогресс и контролируемый полный запуск

Чекпоинт сохраняется атомарно после каждой завершённой эпохи selection/refit. Он содержит контракт, fold/phase/epoch, best epoch, модель, optimizer, Python/NumPy/Torch/DataLoader RNG и накопленный runtime. После resume выводится два времени: текущая сессия и накопленный runtime. Ошибка прекращает run до публикации финальных artifacts; при успешном завершении JSON, NPZ и summary проверяются до удаления checkpoint.

Полный запуск выполняется только явным вызовом `run_stage8()`; эта ячейка сама его не запускает.


In [4]:
import time
from IPython.display import Markdown, display
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit

def runtime_panel(state: CheckpointState, started: float) -> str:
    accumulated = state["_runtime_base"] + (time.monotonic() - started)
    return (
        f"fold={state.get('fold', 0)}/3 phase={state.get('phase', 'init')} "
        f"epoch={state.get('epoch', 0)} | сессия={time.monotonic()-started:.1f}с "
        f"| накопленно={accumulated:.1f}с"
    )

def show_progress(state: CheckpointState, best_auc: float, stale: int, maximum: int, session_started: float) -> None:
    now = time.monotonic(); phase_seconds = float(state.get("phase_runtime_before", 0.0)) + now - float(state["phase_started"])
    fold_seconds = float(state.get("fold_runtime_before", 0.0)) + now - float(state["fold_started"])
    session_seconds = now - session_started; total_seconds = float(state["_runtime_base"]) + session_seconds
    average = phase_seconds / max(int(state["epoch"]), 1); eta = max(maximum - int(state["epoch"]), 0) * average
    checkpoint = state.get("last_checkpoint", "ещё не создан")
    display(Markdown(f"### Прогресс Stage 8 — {state.get('run_mode', 'fresh')}\nВнешний fold {int(state['fold']) + 1}/3 · фаза: {state['phase']} · эпоха {int(state['epoch'])}/{maximum} ({int(state['epoch']) / maximum:.1%})\n\nЛучшая эпоха: {int(state.get('best_epoch', 0))} · лучший inner ROC-AUC: {best_auc:.6f} · patience {stale}/16\n\nФаза: {phase_seconds:.1f}с · fold: {fold_seconds:.1f}с · сессия: {session_seconds:.1f}с · накопленно: {total_seconds:.1f}с\n\nСредняя эпоха: {average:.2f}с · ETA: {eta:.1f}с · последний checkpoint: {checkpoint}") , display_id="stage8_progress", update=True)

def save_checkpoint(state: CheckpointState) -> None:
    save_started = time.monotonic(); now = save_started
    state["fold_runtime_before"] = float(state["fold_runtime_before"]) + now - float(state["fold_started"])
    if state["phase"] in {"selection", "refit"}:
        state["phase_runtime_before"] = float(state["phase_runtime_before"]) + now - float(state["phase_started"])
    state["fold_started"] = now
    state["runtime_before_session"] = state["_runtime_base"] + now - state["_runtime_session_started"]
    state["last_checkpoint"] = {"fold": int(state["fold"]) + 1, "phase": state["phase"], "epoch": int(state["epoch"]), "saved_at": time.strftime("%H:%M:%S")}
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=CHECKPOINT_PATH.parent, delete=False, suffix=".pt") as handle:
        temp_name = handle.name
    torch.save({key: value for key, value in state.items() if not key.startswith("_")}, temp_name)
    os.replace(temp_name, CHECKPOINT_PATH)
    save_finished = time.monotonic(); save_seconds = save_finished - save_started
    # Save входит в fold, но не в training phase; следующий training-интервал начинается после save.
    state["fold_runtime_before"] = float(state["fold_runtime_before"]) + save_seconds
    state["fold_started"] = save_finished
    if state["phase"] in {"selection", "refit"}:
        state["phase_started"] = save_finished
    state["runtime_before_session"] = state["_runtime_base"] + save_finished - state["_runtime_session_started"]
    state["runtime_breakdown"]["phases"]["save"] = state["runtime_breakdown"]["phases"].get("save", 0.0) + save_seconds

def load_checkpoint(contract: Mapping[str, object]) -> CheckpointState:
    if not CHECKPOINT_PATH.exists():
        return {"contract": contract, "fold": 0, "phase": "selection", "epoch": 0,
                "runtime_before_session": 0.0, "fold_results": [], "oof": np.full(len(y_working), np.nan),
                "best_auc": float("-inf"), "best_epoch": 0, "stale": 0,
                "model": {}, "optimizer": {},
                "rng": {"python": random.getstate(), "numpy": np.random.get_state(),
                        "torch": torch.get_rng_state(), "loader": torch.Generator().get_state()},
                "runtime_breakdown": {"folds": {}, "phases": {}}, "phase_runtime_before": 0.0, "fold_runtime_before": 0.0,
                "phase_started": 0.0, "fold_started": 0.0, "run_mode": "fresh", "last_checkpoint": "ещё не создан",
                "_runtime_base": 0.0, "_runtime_session_started": 0.0}
    state: CheckpointState = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    if state["contract"] != contract:
        raise RuntimeError("STOP: контракт checkpoint не совпадает с зафиксированной конфигурацией Stage 8.")
    return state

def rng_state(loader_generator: torch.Generator) -> RngState:
    return {
        "python": random.getstate(), "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(), "loader": loader_generator.get_state(),
    }

def restore_rng(saved: RngState, loader_generator: torch.Generator) -> None:
    random.setstate(saved["python"]); np.random.set_state(saved["numpy"])
    torch.set_rng_state(saved["torch"]); loader_generator.set_state(saved["loader"])

def render_final_section(result) -> None:
    folds = result["outer_folds"]; runtime = result["runtime_breakdown"]
    facts = [f"- Статус: {result['status']}.", f"- B*: {result['B_star']}; метрики: {result['baseline_oof_metrics']}.", f"- FT-Transformer full OOF: {result['ft_transformer_oof_metrics']}.", f"- Full OOF deltas: {result['delta_vs_B_star']}.", f"- Fold deltas: {result['fold_deltas']}.", f"- Лучшие эпохи: {[item['best_epoch'] for item in folds]}.", f"- Runtime: folds={runtime['folds']}; phases={runtime['phases']}; total={runtime['total_seconds']:.1f}с.", f"- Decision: {result['decision']}."]
    for item in folds:
        fm = item['ft_metrics']; facts.append(f"- Fold {item['fold']}: ROC-AUC {fm['roc_auc']:.6f}; Gini {fm['gini']:.6f}; PR-AUC {fm['pr_auc']:.6f}; Precision {fm['precision']:.6f}; Recall {fm['recall']:.6f}; F1 {fm['f1']:.6f}; B* Gini {item['baseline_metrics']['gini']:.6f}; ΔGini {item['delta']['gini']:+.6f}; best_epoch {item['best_epoch']}; runtime fold {runtime['folds'].get(str(item['fold']), 0.0):.1f}с.")
    interpretation = {"material_gain": "Locked rule зафиксировало material gain относительно B*.", "inferior": "Locked rule зафиксировало inferior относительно B*.", "no_material_benefit": "Locked rule не зафиксировало material gain или inferior."}[result['decision']]
    limitations = '\n'.join(f"- {item}" for item in LIMITATIONS)
    next_question = {"material_gain": "Подтверждается ли material gain FT-Transformer на temporal validation без изменения locked feature contract?", "inferior": "Есть ли основания отказаться от FT-Transformer в этом locked design и перейти к следующему заранее определённому семейству моделей?", "no_material_benefit": "Подтверждается ли отсутствие material gain FT-Transformer на независимой temporal validation при том же locked design?"}[result['decision']]
    display(Markdown(f"# Результат исследования\n\n## ФАКТЫ\n{chr(10).join(facts)}\n\n## ИНТЕРПРЕТАЦИЯ\n{interpretation}\n\n## ОГРАНИЧЕНИЯ\n{limitations}\n\n## СЛЕДУЮЩИЙ ШАГ\n{next_question}"))

def run_stage8():
    contract = {
        "stage": "Stage 8 V1", "features": features, "feature_sha256": sha256_bytes('\n'.join(features).encode()),
        "dataset_sha256": stage1["dataset"]["sha256"], "working_index_sha256": sha256_bytes(working_index.tobytes()),
        "outer_cv": {"n_splits": 3, "shuffle": True, "random_state": 42},
        "ft_transformer": FT_CONFIG, "optimizer": {"name": "AdamW", "lr": 1e-4, "weight_decay": 1e-5, "betas": [0.9, 0.999], "eps": 1e-8},
        "preprocessing": PREPROCESSING_CONTRACT, "selection": SELECTION_CONTRACT, "seeds": list(OUTER_FOLD_SEEDS),
        "baseline": {"name": "GBDT_mean", "source": str(STAGE7_OOF_PATH), "retrained": False},
        "final_test_used": False,
    }
    state = load_checkpoint(contract)
    session_started = time.monotonic()
    state["_runtime_base"] = float(state["runtime_before_session"])
    state["run_mode"] = "resume" if CHECKPOINT_PATH.exists() else "fresh"
    state.setdefault("runtime_breakdown", {"folds": {}, "phases": {}})
    state["fold_started"] = session_started
    state["_runtime_session_started"] = session_started
    outer = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    try:
        for fold_number, (outer_train, outer_valid) in enumerate(outer.split(x_working, y_working), start=1):
            if fold_number <= state["fold"]:
                continue
            # Снимок checkpoint ДО любых мутаций state: основа fresh/resume state machine.
            resume_available = CHECKPOINT_PATH.exists()
            resume_fold = int(state["fold"])
            resume_phase = str(state["phase"])
            resume_epoch = int(state["epoch"])
            resumed_selection = resume_available and resume_fold == fold_number - 1 and resume_phase == "selection" and resume_epoch > 0
            resumed_refit = resume_available and resume_fold == fold_number - 1 and resume_phase == "refit" and resume_epoch > 0
            state["fold_started"] = time.monotonic()
            if not (resumed_selection or resumed_refit): state["fold_runtime_before"] = 0.0
            seed = OUTER_FOLD_SEEDS[fold_number - 1]
            split = StratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=seed)
            inner_train_rel, inner_valid_rel = next(split.split(outer_train, y_working[outer_train]))
            inner_train, inner_valid = outer_train[inner_train_rel], outer_train[inner_valid_rel]

            state["phase"] = "selection"
            if not (resumed_selection or resumed_refit): state["phase_runtime_before"] = 0.0
            # Fresh selection: seed фиксируется до model, optimizer и DataLoader.
            set_seed(seed)
            preprocessing_started = time.monotonic()
            x_inner_train, x_inner_valid, transformer = quantile_fit_transform(
                x_working[inner_train], x_working[inner_valid], seed
            )
            state["runtime_breakdown"]["phases"]["preprocessing_selection"] = state["runtime_breakdown"]["phases"].get("preprocessing_selection", 0.0) + time.monotonic() - preprocessing_started
            model = make_model()
            optimizer = make_optimizer(model)
            loader, generator = loader_for(x_inner_train, y_working[inner_train], seed)
            best_auc, best_epoch, stale = -np.inf, 0, 0
            selection_epoch = 0
            if resumed_refit:
                best_auc, best_epoch, stale, selection_epoch = state["best_auc"], state["best_epoch"], 16, int(FT_CONFIG["max_epochs"])
            elif resumed_selection:
                selection_epoch = resume_epoch; best_auc = state["best_auc"]
                best_epoch = state["best_epoch"]; stale = state["stale"]
                model.load_state_dict(state["model"]); optimizer = restore_optimizer_state(model, state["optimizer"])
                restore_rng(state["rng"], generator)
            else:
                pass

            # Training начинается после preprocessing/model setup: интервалы preprocessing, training и save не пересекаются.
            state["phase_started"] = time.monotonic()
            while selection_epoch < int(FT_CONFIG["max_epochs"]) and stale < 16:
                train_one_epoch(model, optimizer, loader)
                selection_epoch += 1
                valid_auc = metrics(y_working[inner_valid], predict(model, x_inner_valid))["roc_auc"]
                if valid_auc > best_auc:
                    best_auc, best_epoch, stale = valid_auc, selection_epoch, 0
                else:
                    stale += 1
                state.update({"fold": fold_number - 1, "phase": "selection", "epoch": selection_epoch,
                              "best_auc": best_auc, "best_epoch": best_epoch, "stale": stale,
                              "model": model.state_dict(), "optimizer": optimizer_state_by_name(model, optimizer),
                              "rng": rng_state(generator)})
                save_checkpoint(state)
                show_progress(state, best_auc, stale, int(FT_CONFIG["max_epochs"]), session_started)

            # При resume refit phase_runtime_before принадлежит только refit и не может попасть в selection evidence.
            selection_seconds = float(state["runtime_breakdown"]["phases"].get(f"fold_{fold_number}_selection", 0.0)) if resumed_refit else float(state["phase_runtime_before"]) + time.monotonic() - float(state["phase_started"])
            state["runtime_breakdown"]["phases"][f"fold_{fold_number}_selection"] = selection_seconds
            if best_epoch < 1:
                raise RuntimeError("STOP: best_epoch не определён.")
            state["phase"] = "refit"
            if not resumed_refit: state["phase_runtime_before"] = 0.0
            # Fresh refit: отдельный детерминированный путь до model, optimizer и DataLoader.
            set_seed(seed)
            preprocessing_refit_started = time.monotonic()
            x_refit, x_outer_valid, _ = quantile_fit_transform(x_working[outer_train], x_working[outer_valid], seed)
            state["runtime_breakdown"]["phases"]["preprocessing_refit"] = state["runtime_breakdown"]["phases"].get("preprocessing_refit", 0.0) + time.monotonic() - preprocessing_refit_started
            refit_model = make_model()
            refit_optimizer = make_optimizer(refit_model)
            refit_loader, refit_generator = loader_for(x_refit, y_working[outer_train], seed)
            refit_epoch = 0
            if resumed_refit:
                refit_epoch = resume_epoch; refit_model.load_state_dict(state["model"])
                refit_optimizer = restore_optimizer_state(refit_model, state["optimizer"])
                restore_rng(state["rng"], refit_generator)
            else:
                pass

            # Refit training начинается после собственного preprocessing/model setup.
            state["phase_started"] = time.monotonic()
            while refit_epoch < best_epoch:
                train_one_epoch(refit_model, refit_optimizer, refit_loader)
                refit_epoch += 1
                state.update({"fold": fold_number - 1, "phase": "refit", "epoch": refit_epoch,
                              "best_epoch": best_epoch, "model": refit_model.state_dict(),
                              "optimizer": optimizer_state_by_name(refit_model, refit_optimizer), "rng": rng_state(refit_generator)})
                save_checkpoint(state)
                show_progress(state, best_auc, 0, best_epoch, session_started)

            refit_seconds = float(state["phase_runtime_before"]) + time.monotonic() - float(state["phase_started"])
            state["runtime_breakdown"]["phases"][f"fold_{fold_number}_refit"] = refit_seconds
            # Fold runtime закрывается только после prediction, metrics и финального fold checkpoint.
            prediction_started = time.monotonic()
            probability = predict(refit_model, x_outer_valid).astype(np.float64)
            state["runtime_breakdown"]["phases"]["prediction"] = state["runtime_breakdown"]["phases"].get("prediction", 0.0) + time.monotonic() - prediction_started
            state["oof"][outer_valid] = probability
            metrics_started = time.monotonic()
            ft_fold_metrics = metrics(y_working[outer_valid], probability)
            baseline_fold_metrics = metrics(y_working[outer_valid], p_baseline[outer_valid])
            fold_delta = {key: ft_fold_metrics[key] - baseline_fold_metrics[key] for key in ft_fold_metrics}
            state["runtime_breakdown"]["phases"]["metrics"] = state["runtime_breakdown"]["phases"].get("metrics", 0.0) + time.monotonic() - metrics_started
            state["fold_results"].append({
                "fold": fold_number, "seed": seed, "best_epoch": best_epoch,
                "selection_roc_auc": best_auc, "ft_metrics": ft_fold_metrics,
                "baseline_metrics": baseline_fold_metrics, "delta": fold_delta,
            })
            state.update({"fold": fold_number, "phase": "completed", "epoch": 0})
            save_checkpoint(state)
            state["runtime_breakdown"]["folds"][str(fold_number)] = float(state["fold_runtime_before"])
    except Exception:
        print("STOP: run прерван; checkpoint сохранён, финальные Stage 8 artifacts не опубликованы.")
        raise

    if not np.isfinite(state["oof"]).all():
        raise RuntimeError("STOP: OOF prediction не заполнен.")
    ft_metrics, baseline_metrics = metrics(y_working, state["oof"]), metrics(y_working, p_baseline)
    delta = {key: ft_metrics[key] - baseline_metrics[key] for key in ft_metrics}
    if not isinstance(state["fold_results"], list): raise RuntimeError("STOP: checkpoint fold results повреждён.")
    fold_gini_deltas = []
    for raw_item in state["fold_results"]:
        if not isinstance(raw_item, Mapping): raise RuntimeError("STOP: checkpoint fold delta повреждён.")
        raw_delta = raw_item.get("delta")
        if not isinstance(raw_delta, Mapping): raise RuntimeError("STOP: checkpoint fold delta повреждён.")
        raw_gini = raw_delta.get("gini")
        if not isinstance(raw_gini, (int, float)): raise RuntimeError("STOP: checkpoint fold delta повреждён.")
        fold_gini_deltas.append(float(raw_gini))
    positive_folds, negative_folds = sum(value > 0.0 for value in fold_gini_deltas), sum(value < 0.0 for value in fold_gini_deltas)
    if delta["gini"] >= 0.010 and positive_folds >= 2:
        decision = "material_gain"
    elif delta["gini"] <= -0.010 and negative_folds >= 2:
        decision = "inferior"
    else:
        decision = "no_material_benefit"
    result = {
        "experiment": "Stage 8", "version": "V1", "stage": "Stage 8 V1 — FT-Transformer vs GBDT_mean", "status": "completed",
        "dataset_sha256": stage1["dataset"]["sha256"], "working_index_sha256": sha256_bytes(working_index.tobytes()),
        "feature_identity": {"features_in_order": features, "sha256": sha256_bytes('\n'.join(features).encode())},
        "training_contract": contract, "preprocessing_contract": PREPROCESSING_CONTRACT, "outer_folds": state["fold_results"],
        "ft_transformer_oof_metrics": ft_metrics, "baseline_oof_metrics": baseline_metrics,
        "B_star": "GBDT_mean", "fold_deltas": {"gini": fold_gini_deltas, "positive_count": positive_folds, "negative_count": negative_folds},
        "delta_vs_B_star": delta, "decision": decision,
        "runtime_seconds": state["runtime_before_session"], "runtime_breakdown": {"total_seconds": state["runtime_before_session"], "folds": state["runtime_breakdown"]["folds"], "phases": state["runtime_breakdown"]["phases"]}, "final_test_used": False,
        "environment": {"python": __import__('sys').version, "numpy": np.__version__, "scikit_learn": __import__('sklearn').__version__, "torch": torch.__version__, "rtdl_revisiting_models": __import__('rtdl_revisiting_models').__version__, "platform": __import__('platform').platform(), "os": os.name, "cpu": __import__('platform').processor(), "cpu_count": os.cpu_count(), "device": "cpu", "torch_threads": torch.get_num_threads(), "num_workers": 0},
        "limitations": LIMITATIONS,
    }
    final_artifact_started = time.monotonic()
    fold_array = np.full(len(y_working), -1, dtype=np.int8)
    for raw_item, (_, outer_valid) in zip(state["fold_results"], StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(x_working, y_working)):
        if not isinstance(raw_item, Mapping): raise RuntimeError("STOP: checkpoint fold повреждён.")
        raw_fold = raw_item.get("fold")
        if not isinstance(raw_fold, int): raise RuntimeError("STOP: checkpoint fold повреждён.")
        fold_array[outer_valid] = raw_fold
    atomic_npz(OOF_PATH, working_indices=working_index, target=y_working, fold=fold_array, ft_transformer_oof_probability=state["oof"], gbdt_mean_probability=p_baseline)
    # Первый атомарный проход создаёт artifacts для проверки; окончательные runtime значения фиксируются ниже.
    atomic_json(RESULT_PATH, result)
    atomic_json(SUMMARY_PATH, {"status": result["status"], "B_star": "GBDT_mean", "oof_metrics": ft_metrics, "deltas": delta, "runtime": result["runtime_breakdown"], "decision": decision, "final_test_used": False})
    if not (RESULT_PATH.exists() and OOF_PATH.exists() and SUMMARY_PATH.exists()):
        raise RuntimeError("STOP: финальные artifacts не прошли проверку существования.")
    with RESULT_PATH.open("r", encoding="utf-8") as handle: json.load(handle)
    with np.load(OOF_PATH) as loaded_oof: _ = loaded_oof["ft_transformer_oof_probability"].shape
    with SUMMARY_PATH.open("r", encoding="utf-8") as handle: json.load(handle)
    CHECKPOINT_PATH.unlink(missing_ok=True)
    # Непрерывный final save interval завершён после NPZ, JSON, existence и readability checks.
    finalization_seconds = time.monotonic() - final_artifact_started
    state["runtime_breakdown"]["phases"]["save"] = finalization_seconds
    final_total_seconds = state["_runtime_base"] + time.monotonic() - state["_runtime_session_started"]
    result["runtime_seconds"] = final_total_seconds
    result["runtime_breakdown"] = {"total_seconds": final_total_seconds, "folds": state["runtime_breakdown"]["folds"], "phases": state["runtime_breakdown"]["phases"]}
    render_final_section(result)
    # Последняя служебная запись сохраняет final runtime; после неё нет проверок или иной finalization-работы.
    atomic_json(RESULT_PATH, result)
    atomic_json(SUMMARY_PATH, {"status": result["status"], "B_star": "GBDT_mean", "oof_metrics": ft_metrics, "deltas": delta, "runtime": result["runtime_breakdown"], "decision": decision, "final_test_used": False})
    return result

print("Готово к контролируемому полному запуску: вызовите run_stage8() только после отдельного решения.")


Готово к контролируемому полному запуску: вызовите run_stage8() только после отдельного решения.


### Что проверяем?

Синтетический preprocessing без leakage.

### Зачем сейчас?

Проверяем recipe до полного run.

### Как это отвечает на исследовательский вопрос?

Подтверждает корректность входов модели.

### Что остаётся неизменным?

QuantileTransformer и training noise из lock.

## Smoke test 1 — preprocessing без leakage

Это маленький синтетический тест, не Stage 8 ML-run. Он проверяет, что fit QuantileTransformer выполняется на train-copy с шумом, а transform даёт float32 и не создаёт NaN.


In [18]:
def smoke_preprocessing() -> None:
    train = np.arange(1, 24001, dtype=np.float64).reshape(1200, 20)
    valid = np.arange(50001, 50401, dtype=np.float64).reshape(20, 20)
    transformed_train, transformed_valid, transformer = quantile_fit_transform(train, valid, seed=43)
    assert transformer.n_quantiles == 1000
    assert transformed_train.dtype == np.float32 and transformed_valid.dtype == np.float32
    assert np.isfinite(transformed_train).all() and np.isfinite(transformed_valid).all()
    assert np.array_equal(train, np.arange(1, 24001, dtype=np.float64).reshape(1200, 20))
    print("SMOKE PASS: предобработка без leakage, float32, без NaN.")

smoke_preprocessing()


SMOKE PASS: предобработка без leakage, float32, без NaN.


### Что проверяем?

Continuous-vs-resume checkpoint smoke.

### Зачем сейчас?

Проверяем точную детерминированность resume.

### Как это отвечает на исследовательский вопрос?

Подтверждает надёжность длительного OOF run.

### Что остаётся неизменным?

Official model, optimizer, CPU и seeded DataLoader.

## Smoke test 2 — checkpoint/resume

Тест использует синтетические 47-признаковые данные и ту же официальную FTTransformer factory. Он прерывается после двух эпох, восстанавливает model/optimizer/RNG и завершает четыре эпохи. Артефакты Stage 8 при этом не создаются.


In [19]:
def smoke_checkpoint_resume() -> None:
    seed, total_epochs, interrupted_after = 701, 4, 2
    x = np.random.default_rng(seed).normal(size=(32, 47)).astype(np.float32)
    y = np.random.default_rng(seed + 1).integers(0, 2, size=32).astype(np.int8)
    set_seed(seed); continuous = make_model(); continuous_optimizer = make_optimizer(continuous); continuous_loader, _ = loader_for(x, y, seed)
    for _ in range(total_epochs): train_one_epoch(continuous, continuous_optimizer, continuous_loader)
    continuous_weights = {name: value.detach().clone() for name, value in continuous.state_dict().items()}
    continuous_probability = predict(continuous, x)
    set_seed(seed); interrupted = make_model(); interrupted_optimizer = make_optimizer(interrupted); interrupted_loader, interrupted_generator = loader_for(x, y, seed)
    for _ in range(interrupted_after): train_one_epoch(interrupted, interrupted_optimizer, interrupted_loader)
    load_trace: list[dict[str, str]] = []
    def traced_load(target: torch.nn.Module, model_state: Mapping[str, torch.Tensor], target_phase: str, source_phase: str) -> None:
        load_trace.append({"target_phase": target_phase, "source_phase": source_phase})
        target.load_state_dict(model_state)
    with tempfile.TemporaryDirectory() as directory:
        smoke_checkpoint = Path(directory) / "resume.pt"
        atomic_torch(smoke_checkpoint, {"last_epoch": interrupted_after, "model": interrupted.state_dict(), "optimizer": optimizer_state_by_name(interrupted, interrupted_optimizer), "rng": rng_state(interrupted_generator)})
        saved = torch.load(smoke_checkpoint, map_location="cpu", weights_only=False)
        set_seed(seed); resumed = make_model(); traced_load(resumed, saved["model"], "selection_resume", "selection")
        resumed_optimizer = restore_optimizer_state(resumed, saved["optimizer"]); resumed_loader, resumed_generator = loader_for(x, y, seed); restore_rng(saved["rng"], resumed_generator)
        resume_start_epoch = int(saved["last_epoch"]) + 1
        for _ in range(resume_start_epoch, total_epochs + 1): train_one_epoch(resumed, resumed_optimizer, resumed_loader)
    weight_diff = max(float(torch.max(torch.abs(continuous_weights[name] - resumed.state_dict()[name])).item()) for name in continuous_weights)
    prediction_diff = float(np.max(np.abs(continuous_probability - predict(resumed, x))))
    # State-machine evidence: interrupted selection resumes at saved_epoch + 1; completed selection never supplies weights to fresh refit.
    selection_saved_epoch = interrupted_after; selection_resume_start_epoch = resume_start_epoch
    set_seed(seed); fresh_refit = make_model(); fresh_refit_start_epoch = 1
    selection_fingerprint = next(iter(interrupted.state_dict().values())).detach().clone()
    fresh_refit_fingerprint = next(iter(fresh_refit.state_dict().values())).detach().clone()
    if torch.equal(selection_fingerprint, fresh_refit_fingerprint): raise RuntimeError("SMOKE FAIL: fresh refit получил selection weights.")
    # Реальный fresh refit: trace фиксирует каждый вызов load_state_dict и его source phase.
    fresh_refit_optimizer = make_optimizer(fresh_refit); fresh_refit_loader, fresh_refit_generator = loader_for(x, y, seed)
    best_epoch = total_epochs; fresh_refit_completed_before_checkpoint = 0
    for _ in range(interrupted_after): train_one_epoch(fresh_refit, fresh_refit_optimizer, fresh_refit_loader); fresh_refit_completed_before_checkpoint += 1
    refit_saved_epoch = fresh_refit_completed_before_checkpoint
    with tempfile.TemporaryDirectory() as directory:
        refit_checkpoint = Path(directory) / "refit_resume.pt"
        atomic_torch(refit_checkpoint, {"last_epoch": refit_saved_epoch, "model": fresh_refit.state_dict(), "optimizer": optimizer_state_by_name(fresh_refit, fresh_refit_optimizer), "rng": rng_state(fresh_refit_generator), "phase": "refit"})
        refit_saved = torch.load(refit_checkpoint, map_location="cpu", weights_only=False)
        set_seed(seed); resumed_refit = make_model(); traced_load(resumed_refit, refit_saved["model"], "refit_resume", "refit")
        resumed_refit_optimizer = restore_optimizer_state(resumed_refit, refit_saved["optimizer"]); resumed_refit_loader, resumed_refit_generator = loader_for(x, y, seed); restore_rng(refit_saved["rng"], resumed_refit_generator)
        refit_resume_start_epoch = int(refit_saved["last_epoch"]) + 1
        refit_completed_epochs = int(refit_saved["last_epoch"])
        for _ in range(refit_resume_start_epoch, best_epoch + 1): train_one_epoch(resumed_refit, resumed_refit_optimizer, resumed_refit_loader); refit_completed_epochs += 1
    selection_state_loaded_into_fresh_refit = any(item["target_phase"] == "fresh_refit" and item["source_phase"] == "selection" for item in load_trace)
    if selection_state_loaded_into_fresh_refit or fresh_refit_start_epoch != 1 or refit_completed_epochs != best_epoch: raise RuntimeError("SMOKE FAIL: fresh refit contract нарушен.")
    trace_text = ','.join(f"{item['target_phase']}<-{item['source_phase']}" for item in load_trace)
    evidence = [f"timestamp={time.strftime('%Y-%m-%d %H:%M:%S')}", f"saved_selection_epoch={selection_saved_epoch}", f"selection_resume_start_epoch={selection_resume_start_epoch}", f"fresh_refit_start_epoch={fresh_refit_start_epoch}", f"selection_state_loaded_into_fresh_refit={str(selection_state_loaded_into_fresh_refit).lower()}", f"load_trace={trace_text}", f"best_epoch={best_epoch}", f"refit_completed_epochs={refit_completed_epochs}", f"saved_refit_epoch={refit_saved_epoch}", f"refit_resume_start_epoch={refit_resume_start_epoch}", f"max_weight_abs_diff={weight_diff:.1f}", f"max_prediction_abs_diff={prediction_diff:.1f}", "overall_status=PASS"]
    smoke_log = GENERATED / "stage8_ft_transformer_smoke_V1.txt"; smoke_log.write_text('\n'.join(evidence) + '\n', encoding="utf-8")
    print(f"State-machine smoke: selection_resume_start_epoch={selection_resume_start_epoch}; fresh_refit_start_epoch={fresh_refit_start_epoch}; refit_resume_start_epoch={refit_resume_start_epoch}; best_epoch={best_epoch}; refit_completed_epochs={refit_completed_epochs}; max_weight_abs_diff={weight_diff:.1f}; max_prediction_abs_diff={prediction_diff:.1f}")
    if weight_diff != 0.0 or prediction_diff != 0.0: raise RuntimeError("SMOKE FAIL: resumed path не совпал с continuous CPU path.")
    print("SMOKE PASS: continuous/resume и fresh refit state-machine подтверждены.")

smoke_checkpoint_resume()


State-machine smoke: selection_resume_start_epoch=3; fresh_refit_start_epoch=1; refit_resume_start_epoch=3; best_epoch=4; refit_completed_epochs=4; max_weight_abs_diff=0.0; max_prediction_abs_diff=0.0
SMOKE PASS: continuous/resume и fresh refit state-machine подтверждены.


## Финальная интерпретация после фактического run

Эта секция заполняется только результатом `run_stage8()`; числа не восстанавливаются по памяти.

- (B^*) — существующий Stage 7 `GBDT_mean`; weighted averaging запрещён, retrain GBDT не выполняется.
- FT-Transformer сравнивается с (B^*) только по outer-OOF; final test не использован.
- Решение формируется только locked rule по full OOF ΔGini и знаку минимум на 2 из 3 folds.
- Превосходство по одной метрике или на трёх random folds не доказывает business benefit и не доказывает temporal stability.
- Результат относится только к locked Stage 8 FT-Transformer design.


## Запуск полного Stage 8 V1

### Что делает следующая ячейка?

Следующая ячейка — это только безопасная точка запуска уже полностью подготовленного Stage 8.

Она **не содержит ML-логику**, не меняет модель, preprocessing, folds, seeds, checkpoint/resume-механику и не создаёт новый экспериментальный протокол.

### Как она работает?

Перед запуском ячейка проверяет, существует ли уже сохранённый финальный результат Stage 8 со статусом `completed`.

Если результат уже существует:

- повторное обучение не запускается;
- выводится сообщение, что Stage 8 уже завершён;
- отображается сохранённый итоговый результат.

Если завершённого результата ещё нет:

- по наличию checkpoint определяется режим запуска;
- при отсутствии checkpoint выполняется `fresh`-запуск;
- при наличии checkpoint выполняется `resume`;
- функция полного Stage 8 вызывается только один раз.

### Что произойдёт при остановке или ошибке?

Если выполнение остановлено через `KeyboardInterrupt`, checkpoint сохраняется и повторный запуск этой же ячейки продолжит эксперимент по существующей resume-логике.

Если возникает другая ошибка, выводятся её тип и текст. Ошибка не маскируется, checkpoint не удаляется, а незавершённый эксперимент не объявляется завершённым.

### Что остаётся неизменным?

- модель FT-Transformer;
- preprocessing;
- CV;
- seeds;
- selection/refit;
- checkpoint/resume logic;
- runtime accounting;
- artifacts;
- decision rule;
- final test policy.

Иными словами, следующая ячейка только **безопасно запускает или продолжает уже реализованный Stage 8 и защищает завершённый результат от случайного повторного ML-run**.

In [20]:
# ============================================================
# STAGE 8 V1 — фиксация изменения compute budget ДО full run
# ============================================================

import json
import os
import time
from pathlib import Path


AMENDMENT_PATH = (
    GENERATED
    / "stage8_ft_transformer_protocol_amendment_V1.json"
)


# ------------------------------------------------------------
# 1. Проверяем, что поменяли именно epochs, а не preprocessing
# ------------------------------------------------------------

assert int(FT_CONFIG["max_epochs"]) == 100, (
    "STOP: max_epochs должен быть равен 100."
)

assert int(SELECTION_CONTRACT["patience"]) == 16, (
    "STOP: patience должен остаться 16."
)

assert float(SELECTION_CONTRACT["min_delta"]) == 0.0, (
    "STOP: min_delta должен остаться 0.0."
)

assert int(PREPROCESSING_CONTRACT["n_quantiles"]) == 1000, (
    "STOP: n_quantiles должен остаться 1000."
)

assert tuple(OUTER_FOLD_SEEDS) == (43, 44, 45), (
    "STOP: fold seeds должны остаться 43/44/45."
)


# ------------------------------------------------------------
# 2. Защита от случайного перезапуска completed Stage 8
# ------------------------------------------------------------

if RESULT_PATH.exists():

    existing_result = json.loads(
        RESULT_PATH.read_text(encoding="utf-8")
    )

    if (
        isinstance(existing_result, dict)
        and existing_result.get("status") == "completed"
    ):
        raise RuntimeError(
            "STOP: уже существует completed Stage 8 result."
        )


# ------------------------------------------------------------
# 3. Старый checkpoint не используем.
#
# После изменения compute ceiling начинаем чистый FRESH run.
# Ничего не удаляем — старый checkpoint сохраняем как evidence.
# ------------------------------------------------------------

archived_checkpoint = None

if CHECKPOINT_PATH.exists():

    timestamp = time.strftime("%Y%m%d_%H%M%S")

    archived_checkpoint = (
        CHECKPOINT_PATH.parent
        / (
            "stage8_ft_transformer_checkpoint_V1_"
            f"before_100ep_{timestamp}.pt"
        )
    )

    CHECKPOINT_PATH.replace(
        archived_checkpoint
    )


# ------------------------------------------------------------
# 4. Сохраняем причину изменения протокола
# ------------------------------------------------------------

amendment = {
    "status": "pre_run_protocol_amendment",
    "stage": "Stage 8 V1",
    "model": "FT-Transformer",

    "created_at": time.strftime(
        "%Y-%m-%d %H:%M:%S"
    ),

    "change": {
        "parameter": "max_epochs",
        "before": 1000,
        "after": 100,
    },

    "reason": (
        "Compute-budget ceiling уменьшен до 100 эпох "
        "для CPU-only запуска до получения Stage 8 OOF-result. "
        "Изменение не основано на качестве FT-Transformer "
        "на данных KOMUS."
    ),

    "unchanged": {
        "features": 47,
        "target": "DefMark",
        "identifier": "INN",
        "patience": 16,
        "min_delta": 0.0,
        "outer_folds": 3,
        "outer_seed": 42,
        "fold_seeds": [43, 44, 45],
        "n_quantiles": 1000,
        "batch_size": 256,
        "final_test_used": False,
    },

    "old_checkpoint_archived": (
        str(archived_checkpoint)
        if archived_checkpoint is not None
        else None
    ),

    "new_run_mode": "fresh",
}


AMENDMENT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

temp_path = AMENDMENT_PATH.with_suffix(
    ".json.tmp"
)

temp_path.write_text(
    json.dumps(
        amendment,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

os.replace(
    temp_path,
    AMENDMENT_PATH,
)


# ------------------------------------------------------------
# 5. Проверяем, что evidence реально записан
# ------------------------------------------------------------

saved = json.loads(
    AMENDMENT_PATH.read_text(
        encoding="utf-8"
    )
)

assert saved["change"]["before"] == 1000
assert saved["change"]["after"] == 100
assert saved["unchanged"]["n_quantiles"] == 1000
assert saved["unchanged"]["final_test_used"] is False

assert not CHECKPOINT_PATH.exists(), (
    "STOP: active checkpoint должен отсутствовать перед FRESH run."
)


print()
print("=" * 68)
print("STAGE 8 PRE-RUN — PASS")
print("=" * 68)
print("Эпохи selection:          максимум 100")
print("Early stopping patience:  16")
print("Outer folds:              3")
print("Fold seeds:               43 / 44 / 45")
print("Batch size:               256")
print("Quantile n_quantiles:      1000 — НЕ менялся")
print("Final test:               НЕ используется")
print("Новый запуск:             FRESH")
print(f"Amendment evidence:        {AMENDMENT_PATH}")

if archived_checkpoint is not None:
    print(f"Старый checkpoint:         {archived_checkpoint}")
else:
    print("Старый checkpoint:         отсутствовал")

print("=" * 68)


STAGE 8 PRE-RUN — PASS
Эпохи selection:          максимум 100
Early stopping patience:  16
Outer folds:              3
Fold seeds:               43 / 44 / 45
Batch size:               256
Quantile n_quantiles:      1000 — НЕ менялся
Final test:               НЕ используется
Новый запуск:             FRESH
Amendment evidence:        d:\Projects\komus-credit-risk\reports\generated\stage8_ft_transformer_protocol_amendment_V1.json
Старый checkpoint:         d:\Projects\komus-credit-risk\reports\generated\stage8_ft_transformer_checkpoint_V1_before_100ep_20260826_233053.pt


In [6]:
print("1. checkpoint существует:", CHECKPOINT_PATH.exists())

print("2. начинаю читать checkpoint...")
_test_checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)
print("3. checkpoint прочитан")

print("fold =", _test_checkpoint["fold"] + 1)
print("phase =", _test_checkpoint["phase"])
print("epoch =", _test_checkpoint["epoch"])
print("best_epoch =", _test_checkpoint["best_epoch"])

print("4. проверяю batched predict...")
print("predict =", predict)
print("ГОТОВО")

1. checkpoint существует: True
2. начинаю читать checkpoint...
3. checkpoint прочитан
fold = 2
phase = completed
epoch = 0
best_epoch = 27
4. проверяю batched predict...
predict = <function predict at 0x000001BCFFE91D00>
ГОТОВО


In [5]:
import json
import time
import threading
from IPython.display import HTML, display


# ============================================================
# LIVE MONITOR ДЛЯ УЖЕ СУЩЕСТВУЮЩЕГО STAGE 8
#
# run_stage8() НЕ переписывается.
# Model / preprocessing / CV / seeds / optimizer / decision
# и artifact paths НЕ меняются.
#
# Здесь добавляется только наблюдаемость:
# - heartbeat во время preprocessing;
# - batch X/Y внутри КАЖДОЙ эпохи;
# - epoch X/max;
# - fold / stage / time / ETA / checkpoint.
# ============================================================


# Сохраняем исходные функции один раз.
# Это важно, чтобы повторный запуск этой же ячейки при RESUME
# не создавал wrappers поверх wrappers.
if "_S8_BASE" not in globals():
    _S8_BASE = {
        "load_checkpoint": load_checkpoint,
        "save_checkpoint": save_checkpoint,
        "quantile_fit_transform": quantile_fit_transform,
        "train_one_epoch": train_one_epoch,
        "predict": predict,
    }


_S8_PANEL = None
_S8_PANEL_LOCK = threading.Lock()

_S8 = {
    "state": None,
    "session_started": None,

    "stage": "подготовка",
    "operation": "инициализация",
    "operation_started": None,
    "detail": "",

    "epoch": None,
    "epoch_max": None,

    "batch": None,
    "batch_total": None,
    "batch_avg": None,

    # Полный epoch-cycle:
    # train -> validation -> checkpoint -> show_progress
    "cycle_started": None,
    "cycle_times": {
        "selection": [],
        "refit": [],
    },
    "seen_cycles": set(),

    "last_train_phase": None,

    # Нужны, чтобы корректно показать epoch при resume refit.
    "resume_phase": None,
    "resume_epoch": 0,
}


# ============================================================
# Форматирование времени
# ============================================================

def _s8_time(seconds):
    if seconds is None:
        return "—"

    try:
        seconds = max(float(seconds), 0.0)
    except (TypeError, ValueError):
        return "—"

    if seconds < 60:
        return f"{seconds:.0f} сек"

    minutes = seconds / 60.0

    if minutes < 60:
        return f"{minutes:.1f} мин"

    hours = int(minutes // 60)
    minutes_left = int(round(minutes - hours * 60))

    if minutes_left == 60:
        hours += 1
        minutes_left = 0

    return f"{hours} ч {minutes_left:02d} мин"


# ============================================================
# Какой outer fold сейчас реально выполняется
# ============================================================

def _s8_current_fold(state):
    if not isinstance(state, dict):
        return 1

    raw_fold = int(state.get("fold", 0))
    phase = str(state.get("phase", ""))

    # Во время training state["fold"] хранит число уже завершённых folds.
    # После state phase="completed" там уже номер завершённого fold.
    if phase == "completed":
        return min(max(raw_fold, 1), 3)

    return min(max(raw_fold + 1, 1), 3)


def _s8_completed_folds(state):
    if not isinstance(state, dict):
        return 0

    return min(
        max(int(state.get("fold", 0)), 0),
        3,
    )


# ============================================================
# Runtime для панели
# ============================================================

def _s8_runtime_values():
    now = time.monotonic()

    state = _S8.get("state")
    session_started = _S8.get("session_started")

    session_seconds = (
        max(now - session_started, 0.0)
        if session_started is not None
        else 0.0
    )

    if isinstance(state, dict):

        accumulated_before_session = float(
            state.get(
                "_runtime_base",
                state.get(
                    "runtime_before_session",
                    0.0,
                ),
            )
        )

        total_seconds = (
            accumulated_before_session
            + session_seconds
        )

        try:
            fold_seconds = (
                float(
                    state.get(
                        "fold_runtime_before",
                        0.0,
                    )
                )
                + max(
                    now
                    - float(
                        state.get(
                            "fold_started",
                            now,
                        )
                    ),
                    0.0,
                )
            )
        except (TypeError, ValueError):
            fold_seconds = 0.0

    else:

        total_seconds = session_seconds
        fold_seconds = session_seconds

    operation_started = _S8.get(
        "operation_started"
    )

    operation_seconds = (
        max(
            now - operation_started,
            0.0,
        )
        if operation_started is not None
        else 0.0
    )

    return (
        operation_seconds,
        fold_seconds,
        session_seconds,
        total_seconds,
    )


# ============================================================
# Среднее фактическое время уже завершённых folds
# ============================================================

def _s8_average_completed_fold(state):
    if not isinstance(state, dict):
        return None

    runtime_breakdown = state.get(
        "runtime_breakdown",
        {},
    )

    if not isinstance(
        runtime_breakdown,
        dict,
    ):
        return None

    fold_map = runtime_breakdown.get(
        "folds",
        {},
    )

    if not isinstance(fold_map, dict):
        return None

    values = []

    for value in fold_map.values():

        try:
            value = float(value)
        except (TypeError, ValueError):
            continue

        if value > 0:
            values.append(value)

    if not values:
        return None

    return sum(values) / len(values)


# ============================================================
# ETA
# ============================================================

def _s8_eta():
    state = _S8.get("state")

    if not isinstance(state, dict):
        return None, None, None

    phase = str(
        state.get(
            "phase",
            "",
        )
    )

    current_fold = _s8_current_fold(
        state
    )

    (
        operation_seconds,
        fold_seconds,
        _,
        _,
    ) = _s8_runtime_values()

    epoch = _S8.get("epoch")
    epoch_max = _S8.get("epoch_max")

    batch = _S8.get("batch")
    batch_total = _S8.get(
        "batch_total"
    )

    batch_avg = _S8.get(
        "batch_avg"
    )

    # --------------------------------------------------------
    # До конца ТЕКУЩЕЙ эпохи.
    #
    # Это появляется уже внутри первой эпохи,
    # после первого/нескольких batches.
    # --------------------------------------------------------

    epoch_eta = None
    projected_epoch_seconds = None

    if (
        batch_avg is not None
        and batch is not None
        and batch_total
    ):

        epoch_eta = (
            max(
                batch_total - batch,
                0,
            )
            * batch_avg
        )

        projected_epoch_seconds = max(
            batch_total * batch_avg,
            operation_seconds,
        )

    # --------------------------------------------------------
    # Средний полный epoch-cycle.
    #
    # После первой завершённой эпохи здесь уже фактическое:
    # train + validation + checkpoint.
    # --------------------------------------------------------

    cycle_history = (
        _S8["cycle_times"]
        .get(
            phase,
            [],
        )
    )

    if cycle_history:

        average_cycle = (
            sum(cycle_history)
            / len(cycle_history)
        )

    else:

        # Пока первая эпоха ещё идёт —
        # используем projection по batches.
        average_cycle = (
            projected_epoch_seconds
        )

    # --------------------------------------------------------
    # ETA текущей крупной стадии
    # --------------------------------------------------------

    stage_eta = None
    current_fold_eta = None

    if (
        phase == "selection"
        and average_cycle is not None
    ):

        stale = int(
            state.get(
                "stale",
                0,
            )
        )

        patience_left = max(
            16 - stale,
            0,
        )

        # Это НЕ обещание окончания.
        # Оценка верна только если ROC-AUC
        # больше не будет улучшаться.
        stage_eta = (
            (epoch_eta or 0.0)
            + patience_left
            * average_cycle
        )

        best_epoch = max(
            int(
                state.get(
                    "best_epoch",
                    0,
                )
            ),
            1,
        )

        # Очень грубая оценка будущего refit.
        projected_refit = (
            best_epoch
            * average_cycle
        )

        current_fold_eta = (
            stage_eta
            + projected_refit
        )

    elif (
        phase == "refit"
        and average_cycle is not None
        and epoch is not None
        and epoch_max
    ):

        remaining_after_current = max(
            int(epoch_max)
            - int(epoch),
            0,
        )

        stage_eta = (
            (epoch_eta or 0.0)
            + remaining_after_current
            * average_cycle
        )

        current_fold_eta = stage_eta

    # --------------------------------------------------------
    # ETA всего Stage 8
    # --------------------------------------------------------

    overall_eta = None

    if current_fold_eta is not None:

        future_full_folds = max(
            3 - current_fold,
            0,
        )

        average_completed_fold = (
            _s8_average_completed_fold(
                state
            )
        )

        if average_completed_fold is None:

            # Пока fold 1 не завершён,
            # используем текущий fold
            # как очень грубый шаблон.
            estimated_fold_total = (
                fold_seconds
                + current_fold_eta
            )

            overall_eta = (
                current_fold_eta
                + future_full_folds
                * estimated_fold_total
            )

        else:

            overall_eta = (
                current_fold_eta
                + future_full_folds
                * average_completed_fold
            )

    return (
        epoch_eta,
        stage_eta,
        overall_eta,
    )


# ============================================================
# Одна компактная панель как в Stage 7
# ============================================================

def _s8_render():
    global _S8_PANEL

    with _S8_PANEL_LOCK:

        state = _S8.get("state")

        (
            operation_seconds,
            fold_seconds,
            session_seconds,
            total_seconds,
        ) = _s8_runtime_values()

        (
            epoch_eta,
            stage_eta,
            overall_eta,
        ) = _s8_eta()

        # ----------------------------------------------------
        # State
        # ----------------------------------------------------

        if isinstance(state, dict):

            current_fold = (
                _s8_current_fold(
                    state
                )
            )

            completed_folds = (
                _s8_completed_folds(
                    state
                )
            )

            phase = str(
                state.get(
                    "phase",
                    "selection",
                )
            )

            best_epoch = int(
                state.get(
                    "best_epoch",
                    0,
                )
            )

            best_auc = state.get(
                "best_auc",
                float("-inf"),
            )

            stale = int(
                state.get(
                    "stale",
                    0,
                )
            )

            run_mode = str(
                state.get(
                    "run_mode",
                    "fresh",
                )
            ).upper()

            checkpoint = state.get(
                "last_checkpoint",
                "ещё не создан",
            )

        else:

            current_fold = 1
            completed_folds = 0
            phase = "подготовка"

            best_epoch = 0
            best_auc = float("-inf")
            stale = 0

            run_mode = (
                "RESUME"
                if CHECKPOINT_PATH.exists()
                else "FRESH"
            )

            checkpoint = (
                "ещё не создан"
            )

        # ----------------------------------------------------
        # Best AUC
        # ----------------------------------------------------

        try:

            best_auc_value = float(
                best_auc
            )

            if (
                best_auc_value
                > float("-inf")
            ):

                best_auc_text = (
                    f"{best_auc_value:.6f}"
                )

            else:

                best_auc_text = (
                    "пока нет"
                )

        except (TypeError, ValueError):

            best_auc_text = (
                "пока нет"
            )

        # ----------------------------------------------------
        # Epoch
        # ----------------------------------------------------

        epoch = _S8.get("epoch")
        epoch_max = _S8.get(
            "epoch_max"
        )

        if epoch is None:

            epoch_text = "—"

        elif epoch_max:

            epoch_text = (
                f"{epoch}/{epoch_max}"
            )

        else:

            epoch_text = str(epoch)

        # ----------------------------------------------------
        # Atomic units = batches
        # ----------------------------------------------------

        batch = _S8.get("batch")
        batch_total = _S8.get(
            "batch_total"
        )

        if (
            batch is not None
            and batch_total
        ):

            batch_pct = (
                100.0
                * batch
                / batch_total
            )

            atomic_text = (
                f"batch "
                f"{batch}/{batch_total} "
                f"({batch_pct:.1f}%)"
            )

        else:

            atomic_text = "—"

        # ----------------------------------------------------
        # Checkpoint
        # ----------------------------------------------------

        if isinstance(
            checkpoint,
            dict,
        ):

            checkpoint_text = (
                f"fold "
                f"{checkpoint.get('fold', '?')}"
                f" · "
                f"{checkpoint.get('phase', '?')}"
                f" · epoch "
                f"{checkpoint.get('epoch', '?')}"
                f" · "
                f"{checkpoint.get('saved_at', '?')}"
            )

        else:

            checkpoint_text = str(
                checkpoint
            )

        # ----------------------------------------------------
        # Patience
        # ----------------------------------------------------

        if phase == "selection":

            patience_text = (
                f"{stale}/16"
            )

            stage_eta_note = (
                "если ROC-AUC больше "
                "не улучшится"
            )

        else:

            patience_text = "—"
            stage_eta_note = ""

        average_fold = (
            _s8_average_completed_fold(
                state
            )
        )

        average_batch = _S8.get(
            "batch_avg"
        )

        # ----------------------------------------------------
        # HTML
        # ----------------------------------------------------

        html = f"""
        <div style="
            font-family: Arial, sans-serif;
            font-size: 15px;
            line-height: 1.45;
            border: 1px solid #888;
            padding: 11px 14px;
            max-width: 900px;
        ">

            <div style="
                font-size: 18px;
                font-weight: 700;
                margin-bottom: 7px;
            ">
                Этап 8 V1 — FT-Transformer
            </div>

            <b>Режим:</b>
            {run_mode}
            <br>

            <b>Внешний fold:</b>
            {current_fold}/3
            &nbsp;·&nbsp;
            <b>завершено:</b>
            {completed_folds}/3
            <br>

            <b>Крупный этап:</b>
            {_S8.get("stage", "—")}
            <br>

            <b>Текущая операция:</b>
            {_S8.get("operation", "—")}
            <br>

            <b>Эпоха / max:</b>
            {epoch_text}
            <br>

            <b>Atomic units:</b>
            {atomic_text}
            <br>

            <b>Best epoch:</b>
            {best_epoch}
            &nbsp;·&nbsp;
            <b>Лучший inner ROC-AUC:</b>
            {best_auc_text}
            <br>

            <b>Patience:</b>
            {patience_text}
            <br>

            <br>

            <b>Время операции:</b>
            {_s8_time(operation_seconds)}
            &nbsp;·&nbsp;

            <b>fold:</b>
            {_s8_time(fold_seconds)}
            &nbsp;·&nbsp;

            <b>сессии:</b>
            {_s8_time(session_seconds)}
            &nbsp;·&nbsp;

            <b>общее:</b>
            {_s8_time(total_seconds)}
            <br>

            <b>Средний batch:</b>
            {_s8_time(average_batch)}
            <br>

            <b>До конца текущей эпохи:</b>
            {_s8_time(epoch_eta)}
            <br>

            <b>До конца текущего этапа:</b>
            {_s8_time(stage_eta)}

            {
                f"<span style='color:#aaa'>"
                f"({stage_eta_note})"
                f"</span>"
                if stage_eta_note
                else ""
            }

            <br>

            <b>Средний завершённый fold:</b>
            {_s8_time(average_fold)}
            <br>

            <b>Ориентировочно до конца Stage 8:</b>
            {_s8_time(overall_eta)}
            <br>

            <br>

            <b>Последний checkpoint:</b>
            {checkpoint_text}
            <br>

            <span style="color:#aaa;">
                {_S8.get("detail", "")}
                · обновлено
                {time.strftime("%H:%M:%S")}
            </span>

        </div>
        """

        payload = HTML(html)

        if _S8_PANEL is None:

            _S8_PANEL = display(
                payload,
                display_id=True,
            )

        else:

            _S8_PANEL.update(
                payload
            )


# ============================================================
# Начать новую отображаемую операцию
# ============================================================

def _s8_operation(
    stage,
    operation,
    detail="",
    epoch=None,
    epoch_max=None,
    batch=None,
    batch_total=None,
):

    _S8.update({
        "stage": stage,
        "operation": operation,
        "operation_started": (
            time.monotonic()
        ),
        "detail": detail,

        "epoch": epoch,
        "epoch_max": epoch_max,

        "batch": batch,
        "batch_total": batch_total,
        "batch_avg": None,
    })

    _s8_render()


# ============================================================
# Heartbeat для blocking operations
# ============================================================

def _s8_heartbeat(stop_event):
    while not stop_event.wait(30.0):

        try:
            _s8_render()

        except Exception:
            # Панель никогда не должна
            # остановить ML.
            pass


def _s8_blocking(
    stage,
    operation,
    detail,
    func,
    *args,
    **kwargs,
):

    _s8_operation(
        stage,
        operation,
        detail,
    )

    stop_event = threading.Event()

    worker = threading.Thread(
        target=_s8_heartbeat,
        args=(stop_event,),
        daemon=True,
    )

    worker.start()

    try:

        return func(
            *args,
            **kwargs,
        )

    finally:

        stop_event.set()

        worker.join(
            timeout=1.0
        )

        try:
            _s8_render()
        except Exception:
            pass


# ============================================================
# 1. LOAD CHECKPOINT
#
# Только сохраняем ссылку на реальный state,
# который уже использует run_stage8().
# ============================================================

def load_checkpoint(contract):

    state = _S8_BASE["load_checkpoint"](contract)

    _S8["state"] = state

    _S8["resume_phase"] = str(
        state.get("phase", "selection")
    )

    _S8["resume_epoch"] = int(
        state.get("epoch", 0)
    )

    _S8["last_train_phase"] = None

    completed_folds = int(
        state.get("fold", 0)
    )

    if state.get("phase") == "completed":

        next_fold = min(
            completed_folds + 1,
            3,
        )

        _S8.update({
            "stage": "checkpoint загружен",
            "operation": (
                f"Fold {completed_folds} завершён → "
                f"подготовка Fold {next_fold}/3"
            ),
            "operation_started": time.monotonic(),
            "detail": (
                f"checkpoint: phase=completed; "
                f"завершено folds={completed_folds}/3"
            ),
            "epoch": None,
            "epoch_max": None,
            "batch": None,
            "batch_total": None,
            "batch_avg": None,
        })

    else:

        _S8.update({
            "stage": "checkpoint загружен",
            "operation": (
                f"продолжение {state.get('phase')} "
                f"epoch {state.get('epoch')}"
            ),
            "operation_started": time.monotonic(),
            "detail": (
                f"best_epoch={state.get('best_epoch', 0)}"
            ),
            "epoch": int(state.get("epoch", 0)),
            "epoch_max": (
                int(state.get("best_epoch", 0))
                if state.get("phase") == "refit"
                else int(FT_CONFIG["max_epochs"])
            ),
            "batch": None,
            "batch_total": None,
            "batch_avg": None,
        })

    _s8_render()

    return state


# ============================================================
# 2. PREPROCESSING
#
# Сам QuantileTransformer остаётся исходным.
# Мы только показываем, что он работает,
# и обновляем elapsed каждые ~30 секунд.
# ============================================================

def quantile_fit_transform(
    x_train,
    x_apply,
    seed,
):

    state = _S8.get("state")

    if isinstance(state, dict):

        phase = str(
            state.get(
                "phase",
                "selection",
            )
        )

    else:

        phase = "selection"

    if phase == "selection":

        stage = (
            "1/7 — preprocessing selection"
        )

    else:

        stage = (
            "3/7 — preprocessing refit"
        )

    detail = (
        f"train rows={len(x_train):,}; "
        f"apply rows={len(x_apply):,}; "
        f"seed={seed}"
    )

    return _s8_blocking(
        stage,
        "QuantileTransformer",
        detail,
        _S8_BASE[
            "quantile_fit_transform"
        ],
        x_train,
        x_apply,
        seed,
    )


# ============================================================
# 3. LIVE TRAINING
#
# Это единственная функция, где нужен batch-level progress.
#
# Математика ниже совпадает с исходной Stage 8 train_one_epoch:
#
# optimizer.zero_grad
# model(...)
# BCEWithLogits
# backward
# optimizer.step
#
# Никаких новых optimizer steps / clipping / scheduler /
# sampling / weights здесь нет.
# ============================================================

def train_one_epoch(
    model,
    optimizer,
    loader,
):

    state = _S8.get("state")

    phase = str(
        state.get(
            "phase",
            "selection",
        )
    )

    # --------------------------------------------------------
    # Номер реально выполняемой эпохи
    # --------------------------------------------------------

    if phase == "selection":

        current_epoch = (
            int(
                state.get(
                    "epoch",
                    0,
                )
            )
            + 1
        )

        epoch_max = int(FT_CONFIG["max_epochs"])

    else:

        previous_train_phase = (
            _S8.get(
                "last_train_phase"
            )
        )

        # Fresh transition:
        # completed selection -> new refit.
        if previous_train_phase == "selection":

            current_epoch = 1

        # Resume непосредственно refit.
        elif (
            previous_train_phase is None
            and _S8.get(
                "resume_phase"
            )
            == "refit"
        ):

            current_epoch = (
                int(
                    _S8.get(
                        "resume_epoch",
                        0,
                    )
                )
                + 1
            )

        else:

            current_epoch = (
                int(
                    state.get(
                        "epoch",
                        0,
                    )
                )
                + 1
            )

        epoch_max = max(
            int(
                state.get(
                    "best_epoch",
                    0,
                )
            ),
            current_epoch,
        )

    total_batches = len(loader)

    _S8["cycle_started"] = (
        time.monotonic()
    )

    _S8["last_train_phase"] = (
        phase
    )

    if phase == "selection":

        stage = (
            "2/7 — SELECTION"
        )

    else:

        stage = (
            "4/7 — REFIT"
        )

    _s8_operation(
        stage,
        (
            f"обучение эпохи "
            f"{current_epoch}/{epoch_max}"
        ),
        (
            "forward → loss → "
            "backward → optimizer.step"
        ),
        epoch=current_epoch,
        epoch_max=epoch_max,
        batch=0,
        batch_total=total_batches,
    )

    # --------------------------------------------------------
    # Исходная математическая логика Stage 8
    # --------------------------------------------------------

    model.train()

    batch_times = []

    last_update = time.monotonic()

    # Около 100 обновлений максимум на эпоху,
    # но не реже примерно одного раза в 30 сек.
    update_each = max(
        total_batches // 100,
        1,
    )

    for (
        batch_no,
        (batch_x, batch_y),
    ) in enumerate(
        loader,
        start=1,
    ):

        batch_started = (
            time.monotonic()
        )

        # ================================================
        # ИСХОДНАЯ TRAIN МАТЕМАТИКА — НЕ МЕНЯТЬ
        # ================================================

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = (
            model(
                batch_x.to(DEVICE),
                None,
            )
            .squeeze(1)
        )

        loss = (
            F.binary_cross_entropy_with_logits(
                logits,
                batch_y.to(DEVICE),
            )
        )

        loss.backward()

        optimizer.step()

        # ================================================

        batch_times.append(
            time.monotonic()
            - batch_started
        )

        now = time.monotonic()

        if (
            batch_no == 1
            or batch_no == total_batches
            or batch_no % update_each == 0
            or now - last_update >= 30.0
        ):

            _S8["batch"] = batch_no

            _S8["batch_total"] = (
                total_batches
            )

            _S8["batch_avg"] = (
                sum(batch_times)
                / len(batch_times)
            )

            _S8["detail"] = (
                f"epoch "
                f"{current_epoch}/{epoch_max}; "
                f"batch "
                f"{batch_no}/{total_batches}; "
                f"last loss="
                f"{float(loss.detach().cpu()):.6f}"
            )

            _s8_render()

            last_update = now


# ============================================================
# 4. PREDICTION
#
# Исходный predict не переписывается.
# Только heartbeat во время блокирующего inference.
# ============================================================

def predict(
    model,
    x,
):

    state = _S8.get("state")

    phase = (
        str(
            state.get(
                "phase",
                "",
            )
        )
        if isinstance(
            state,
            dict,
        )
        else ""
    )

    if phase == "selection":

        stage = (
            "2/7 — SELECTION / validation"
        )

        operation = (
            "validation prediction "
            "/ inner ROC-AUC"
        )

    else:

        stage = (
            "5/7 — outer prediction"
        )

        operation = (
            "prediction "
            "на outer validation"
        )

    return _s8_blocking(
        stage,
        operation,
        f"rows={len(x):,}",
        _S8_BASE["predict"],
        model,
        x,
    )


# ============================================================
# 5. CHECKPOINT
#
# Исходная save_checkpoint выполняется полностью.
# Мы только показываем её начало/конец.
# ============================================================

def save_checkpoint(state):

    _S8["state"] = state

    phase = str(
        state.get(
            "phase",
            "",
        )
    )

    epoch = int(
        state.get(
            "epoch",
            0,
        )
    )

    if phase == "selection":

        stage = (
            "2/7 — SELECTION / checkpoint"
        )

        epoch_max = int(FT_CONFIG["max_epochs"])

    elif phase == "refit":

        stage = (
            "4/7 — REFIT / checkpoint"
        )

        epoch_max = int(
            state.get(
                "best_epoch",
                0,
            )
        )

    else:

        stage = (
            "7/7 — fold save / завершение"
        )

        epoch_max = None

    _s8_operation(
        stage,
        "atomic checkpoint save",
        (
            f"phase={phase}; "
            f"epoch={epoch}"
        ),
        epoch=(
            epoch
            if phase
            in {
                "selection",
                "refit",
            }
            else None
        ),
        epoch_max=epoch_max,
    )

    stop_event = threading.Event()

    worker = threading.Thread(
        target=_s8_heartbeat,
        args=(stop_event,),
        daemon=True,
    )

    worker.start()

    try:

        _S8_BASE[
            "save_checkpoint"
        ](
            state
        )

    finally:

        stop_event.set()

        worker.join(
            timeout=1.0
        )

    _S8["detail"] = (
        "checkpoint сохранён: "
        f"{state.get('last_checkpoint', '—')}"
    )

    _s8_render()


# ============================================================
# 6. AFTER-EPOCH PROGRESS
#
# Это имя уже вызывает исходный run_stage8()
# после каждого успешного checkpoint.
#
# Здесь фиксируется полный epoch cycle:
# train + validation + checkpoint.
# ============================================================

def show_progress(
    state,
    best_auc,
    stale,
    maximum,
    session_started,
):

    _S8["state"] = state

    phase = str(
        state.get(
            "phase",
            "",
        )
    )

    epoch = int(
        state.get(
            "epoch",
            0,
        )
    )

    fold = _s8_current_fold(
        state
    )

    cycle_started = _S8.get(
        "cycle_started"
    )

    cycle_key = (
        fold,
        phase,
        epoch,
    )

    if (
        phase
        in {
            "selection",
            "refit",
        }
        and cycle_started is not None
        and cycle_key
        not in _S8["seen_cycles"]
    ):

        _S8[
            "cycle_times"
        ][phase].append(
            max(
                time.monotonic()
                - cycle_started,
                0.0,
            )
        )

        _S8["seen_cycles"].add(
            cycle_key
        )

    if phase == "selection":

        stage = (
            "2/7 — SELECTION"
        )

        detail = (
            f"best_epoch="
            f"{int(state.get('best_epoch', 0))}; "
            f"best inner ROC-AUC="
            f"{float(best_auc):.6f}; "
            f"patience="
            f"{int(stale)}/16"
        )

    else:

        stage = (
            "4/7 — REFIT"
        )

        detail = (
            f"refit epoch "
            f"{epoch}/{maximum} "
            f"завершена"
        )

    _S8.update({
        "stage": stage,

        "operation": (
            f"эпоха "
            f"{epoch}/{maximum} "
            f"завершена; "
            f"checkpoint сохранён"
        ),

        "operation_started": (
            time.monotonic()
        ),

        "detail": detail,

        "epoch": epoch,
        "epoch_max": int(maximum),

        "batch": None,
        "batch_total": None,
        "batch_avg": None,
    })

    _s8_render()


# ============================================================
# ВАЖНО:
#
# atomic_json / atomic_npz / RESULT_PATH / OOF_PATH /
# SUMMARY_PATH здесь НЕ переопределяются.
#
# Финальные artifacts продолжает публиковать исходный
# принятый run_stage8() ровно по своей текущей логике.
# ============================================================


# ============================================================
# SAFE LAUNCH / RESUME
# ============================================================

try:

    existing_result = None

    if RESULT_PATH.exists():

        existing_result = (
            json.loads(
                RESULT_PATH.read_text(
                    encoding="utf-8"
                )
            )
        )

    # --------------------------------------------------------
    # Уже завершён
    # --------------------------------------------------------

    if (
        isinstance(
            existing_result,
            dict,
        )
        and existing_result.get(
            "status"
        )
        == "completed"
    ):

        _s8_operation(
            "COMPLETED",
            "Stage 8 уже завершён",
            (
                "повторный ML-run "
                "не запускается"
            ),
        )

        render_final_section(
            existing_result
        )

    # --------------------------------------------------------
    # Fresh / Resume
    # --------------------------------------------------------

    else:

        _S8_PANEL = None

        _S8["session_started"] = (
            time.monotonic()
        )

        mode = (
            "RESUME"
            if CHECKPOINT_PATH.exists()
            else "FRESH"
        )

        _s8_operation(
            "подготовка",
            (
                "загрузка состояния "
                "/ старт run"
            ),
            (
                f"режим={mode}; "
                "heartbeat каждые ~30 сек; "
                "внутри train показывается "
                "каждый ~1% batches"
            ),
        )

        stage8_result = (
            run_stage8()
        )

        _s8_operation(
            "COMPLETED",
            "Stage 8 завершён",
            (
                f"results="
                f"{RESULT_PATH.name}; "
                f"oof="
                f"{OOF_PATH.name}; "
                f"summary="
                f"{SUMMARY_PATH.name}"
            ),
        )


# ============================================================
# Остановка пользователем
# ============================================================

except KeyboardInterrupt:

    if CHECKPOINT_PATH.exists():

        checkpoint_info = (
            "последний завершённый "
            f"checkpoint: "
            f"{CHECKPOINT_PATH.name}"
        )

    else:

        checkpoint_info = (
            "checkpoint ещё не создан; "
            "следующий запуск "
            "начнётся fresh"
        )

    _s8_operation(
        "STOPPED",
        (
            "Stage 8 остановлен "
            "пользователем"
        ),
        (
            f"{checkpoint_info}; "
            "повторный запуск "
            "этой же ячейки "
            "продолжит безопасно"
        ),
    )

    raise


# ============================================================
# Ошибка
# ============================================================

except Exception as exc:

    _s8_operation(
        "ERROR",
        (
            "Stage 8 остановлен "
            "из-за ошибки"
        ),
        (
            f"{type(exc).__name__}: "
            f"{exc}; "
            "существующий checkpoint "
            "не удаляется"
        ),
    )

    raise

# Результат исследования

## ФАКТЫ
- Статус: completed.
- B*: GBDT_mean; метрики: {'roc_auc': 0.9031996976062311, 'gini': 0.8063993952124622, 'pr_auc': 0.6038568965217617, 'precision': 0.7246253285034449, 'recall': 0.3641620560414064, 'f1': 0.48472466384757923}.
- FT-Transformer full OOF: {'roc_auc': 0.9007642154226978, 'gini': 0.8015284308453956, 'pr_auc': 0.593910939146681, 'precision': 0.6863990040460629, 'recall': 0.3936105657683384, 'f1': 0.5003176043557169}.
- Full OOF deltas: {'roc_auc': -0.0024354821835332885, 'gini': -0.004870964367066577, 'pr_auc': -0.009945957375080638, 'precision': -0.038226324457381944, 'recall': 0.029448509726932026, 'f1': 0.015592940508137698}.
- Fold deltas: {'gini': [-0.006121209136584982, -0.0010648902285423922, -0.0007818037180384874], 'positive_count': 0, 'negative_count': 3}.
- Лучшие эпохи: [27, 19, 10].
- Runtime: folds={'2': 45958.923, '3': 27515.296999999984}; phases={'preprocessing_selection': 20.842000000016924, 'save': 1.0460000000020955, 'fold_1_selection': 30345.17199999999, 'preprocessing_refit': 26.95199999999022, 'fold_1_refit': 21323.707000000028, 'prediction': 294.0309999999954, 'metrics': 0.4219999999986612, 'fold_2_selection': 30318.629000000044, 'fold_2_refit': 15529.143000000025, 'fold_3_selection': 19065.529000000028, 'fold_3_refit': 8340.641000000003}; total=125275.4с.
- Decision: no_material_benefit.
- Fold 1: ROC-AUC 0.896832; Gini 0.793663; PR-AUC 0.591722; Precision 0.653045; Recall 0.439816; F1 0.525629; B* Gini 0.799785; ΔGini -0.006121; best_epoch 27; runtime fold 0.0с.
- Fold 2: ROC-AUC 0.903415; Gini 0.806829; PR-AUC 0.599662; Precision 0.668819; Recall 0.421504; F1 0.517112; B* Gini 0.807894; ΔGini -0.001065; best_epoch 19; runtime fold 45958.9с.
- Fold 3: ROC-AUC 0.905457; Gini 0.810914; PR-AUC 0.607994; Precision 0.766898; Recall 0.319520; F1 0.451096; B* Gini 0.811695; ΔGini -0.000782; best_epoch 10; runtime fold 27515.3с.

## ИНТЕРПРЕТАЦИЯ
Locked rule не зафиксировало material gain или inferior.

## ОГРАНИЧЕНИЯ
- random CV не доказывает temporal stability
- 3 folds не являются statistical significance claim
- FT-Transformer не доказывает business benefit
- threshold 0.5 диагностический
- final test не использован
- результат относится только к locked Stage 8 FT-Transformer design

## СЛЕДУЮЩИЙ ШАГ
Подтверждается ли отсутствие material gain FT-Transformer на независимой temporal validation при том же locked design?

# Результат исследования

## ФАКТЫ

Stage 8 V1 завершён успешно.

- статус experiment: `completed`;
- завершены все `3 из 3` outer folds;
- сформирован полный OOF-прогноз;
- OOF artifact содержит `289 614` наблюдений;
- final test не использовался;
- контрольная модель: `B*=GBDT_mean`;
- `GBDT_mean` в Stage 8 заново не обучался.

### Полный OOF

| Метрика | FT-Transformer | GBDT_mean | Δ FT − baseline |
|---|---:|---:|---:|
| ROC-AUC | 0.900764 | 0.903200 | -0.002435 |
| Gini | 0.801528 | 0.806399 | **-0.004871** |
| PR-AUC | 0.593911 | 0.603857 | -0.009946 |
| Precision | 0.686399 | 0.724625 | -0.038226 |
| Recall | 0.393611 | 0.364162 | +0.029449 |
| F1 | 0.500318 | 0.484725 | +0.015593 |

### Fold-level evidence

| Fold | FT Gini | B* Gini | ΔGini | FT Recall | B* Recall | best epoch |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | 0.793663 | 0.799785 | -0.006121 | 0.439816 | 0.361641 | 27 |
| 2 | 0.806829 | 0.807894 | -0.001065 | 0.421504 | 0.365282 | 19 |
| 3 | 0.810914 | 0.811695 | -0.000782 | 0.319520 | 0.365564 | 10 |

По Gini FT-Transformer оказался ниже `GBDT_mean` на всех трёх folds:

`[-0.006121, -0.001065, -0.000782]`.

### Runtime

Полный сохранённый runtime Stage 8:

`125 275.4 сек`, то есть примерно **34 ч 48 мин** CPU-выполнения.

Для Fold 2 сохранено `45 958.9 сек`, для Fold 3 — `27 515.3 сек`.

**Fold 1: отдельный fold-level runtime не сохранился после resume; общий runtime Stage 8 включает выполненную работу Fold 1.**

Не следует восстанавливать или подставлять для Fold 1 выдуманное числовое значение.

### Решение по заранее зафиксированному правилу

Полный:

`ΔGini = -0.004871`.

Это значение:

- не достигает порога `+0.010`, требуемого для `material_gain`;
- не достигает порога `-0.010`, требуемого для `inferior`.

Поэтому зафиксированное решение:

**`no_material_benefit`**.

## ИНТЕРПРЕТАЦИЯ

FT-Transformer **не подтвердил существенного преимущества над `GBDT_mean`** в зафиксированном Stage 8 protocol.

По основной исследовательской метрике Gini его полный OOF-результат немного ниже baseline:

- FT-Transformer: `0.801528`;
- `GBDT_mean`: `0.806399`;
- разница: `-0.004871`.

Кроме того, направление разницы по Gini одинаково на всех трёх folds: FT-Transformer ни на одном из них не превзошёл `GBDT_mean`.

При этом разница недостаточна для заранее определённого статуса `inferior`, поэтому корректный итог — именно `no_material_benefit`, а не утверждение о существенном проигрыше модели.

По вспомогательным метрикам картина неоднозначна.

При диагностическом пороге `0.5` FT-Transformer:

- повысил `Recall` примерно на `2.94` процентного пункта;
- повысил `F1` примерно на `1.56` процентного пункта;
- одновременно снизил `Precision` примерно на `3.82` процентного пункта.

То есть модель при этом пороге находила несколько большую долю объектов дефолтного класса, но делала это ценой большего количества ложноположительных решений.

Это не меняет основной research decision: Stage 8 заранее определён как сравнение по outer-OOF Gini, и преимущества FT-Transformer по этой метрике не получено.

Таким образом, текущий результат **не даёт оснований заменять принятый `GBDT_mean` на FT-Transformer** в рамках данного 47-признакового протокола.

## ОГРАНИЧЕНИЯ

- Используется random cross-validation. Она оценивает качество на случайных разбиениях рабочей выборки, но **не доказывает temporal stability**.
- В имеющихся данных нет подтверждённой надёжной row-level observation date, поэтому из этого Stage нельзя делать вывод о поведении модели во времени.
- Три folds дают воспроизводимое сравнение в текущем protocol, но сами по себе не являются доказательством statistical significance различий.
- `Precision`, `Recall` и `F1` рассчитаны при диагностическом пороге `0.5`. Threshold optimization в Stage 8 не проводился.
- Более высокий Recall при `0.5` сам по себе не доказывает business benefit: политика ошибок FN/FP в этом эксперименте отдельно не исследовалась.
- Проверена одна заранее зафиксированная конфигурация FT-Transformer. Результат нельзя обобщать на все возможные конфигурации или на весь класс Transformer-моделей для табличных данных.
- Эксперимент не доказывает причинное влияние каких-либо признаков на дефолт.
- Final test не использовался. Поэтому он не повлиял на выбор модели, настроек или направление исследования.

## СЛЕДУЮЩИЙ ШАГ

Текущий исследовательский вопрос закрыт:

**в Stage 8 V1 FT-Transformer не показал material gain по outer-OOF Gini относительно `B*=GBDT_mean`.**

При этом эксперимент оставил одну важную неопределённость.

Хотя FT-Transformer уступил baseline по Gini и PR-AUC и показал более низкий Gini на всех трёх folds, при диагностическом пороге `0.5` его Recall оказался выше примерно на `2.94` п.п., а F1 — примерно на `1.56` п.п.

Этот результат нельзя автоматически считать преимуществом FT-Transformer.

Порог `0.5` не является выбранным рабочим threshold, а шкалы score у двух моделей могут различаться. Поэтому одна модель может просто чаще переводить объекты через числовую границу `0.5`, не обеспечивая при этом лучшего ранжирования дефолтных компаний.

### Следующий исследовательский вопрос

**При одинаковой фиксированной доле компаний, отнесённых к high-risk по OOF ranking, действительно ли FT-Transformer лучше `GBDT_mean` захватывает трудные дефолты из фиксированной Stage 3 blind spot, или его более высокий Recall/F1 при threshold `0.5` является в основном следствием другой шкалы score?**

Следующий этап должен сравнивать модели **при одинаковой risk capacity / percentile ranking**, а не при одном числовом cutoff.

Особое внимание нужно уделить заранее определённой Stage 3 blind spot.

Если FT-Transformer при одинаковой capacity действительно поднимает заметную часть этих дефолтов выше в ranking, это будет evidence в пользу реальной model complementarity и даст основание отдельно исследовать простой hybrid-механизм.

Если такого rescue не обнаружится, более высокий Recall при `0.5` будет корректнее трактовать преимущественно как эффект threshold / score scale, а не как подтверждённое преимущество FT-Transformer.

Для этого исследования новое обучение моделей не требуется: предполагается использовать уже сохранённые row-level OOF predictions Stage 3, Stage 7 и Stage 8 после проверки их identity и working-index alignment.

Final test для этого вопроса не используется.